# 20. IF/Math Attribute Importance Distribution Analysis

이 노트북은 `if` / `math` precomputed attribute importance score를 로드한 뒤, 아래 항목을 실험별로 분리해 분석합니다.

1. threshold (`1e-8`, `1e-7`, `1e-6`, `1e-5`) 기준 active parameter 비율
2. threshold별 active mass / total mass 비율
3. 동일 분석을 `sqrt(score)`에 대해 반복
4. 경향을 보기 쉬운 축 범위(로그 x축 + y축 zoom) 기반 시각화

## Experiment 0. Setup

In [ ]:
from __future__ import annotations

import gc
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from matplotlib.ticker import PercentFormatter

try:
    import seaborn as sns

    HAVE_SEABORN = True
except ModuleNotFoundError:
    # Fallback path keeps plotting functional even when seaborn is not installed.
    sns = None
    HAVE_SEABORN = False

# Use a deterministic plotting style so that reruns are visually comparable.
if HAVE_SEABORN:
    sns.set_theme(style="whitegrid", context="talk")
else:
    plt.style.use("default")

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

## Experiment 1. Importance 파일 경로 설정 및 로드 대상 확인

In [ ]:
@dataclass(frozen=True)
class PrecomputedImportanceConfig:
    """Configuration used to resolve precomputed importance file names.

    Args:
        root_dir: Directory that stores serialized importance `.pt` files.
        score_mode: One of `absolute`, `positive`, or `negative`.
        selection_side: One of `top` or `bottom`.
        selection_percent: Selection percentage used in file naming.
    """

    root_dir: Path
    score_mode: str = "absolute"
    selection_side: str = "top"
    selection_percent: float = 15.0


def _normalize_selection_percent(selection_percent: float) -> float:
    """Normalize ratio/percentage selection input into percentage.

    Args:
        selection_percent: Ratio (`0 < x <= 1`) or percentage (`x > 1`).

    Returns:
        Normalized percentage in `(0, inf)`.

    Raises:
        ValueError: If the input is not positive.
    """

    raw_value = float(selection_percent)
    if raw_value <= 0.0:
        raise ValueError(f"selection_percent must be positive, got {raw_value}")
    if raw_value <= 1.0:
        return raw_value * 100.0
    return raw_value


def _format_selection_percent_token(selection_percent: float) -> str:
    """Format percentage into a filename-safe token.

    Args:
        selection_percent: Normalized percentage value.

    Returns:
        Token used in file names, e.g. `15` or `12p5`.
    """

    percent_string = f"{float(selection_percent):.6f}".rstrip("0").rstrip(".")
    return percent_string.replace(".", "p")


def build_precomputed_importance_filename(task_name: str, config: PrecomputedImportanceConfig) -> str:
    """Build a precomputed importance filename for a task.

    Args:
        task_name: Task identifier such as `if` or `math`.
        config: Filename configuration.

    Returns:
        Resolved filename string.
    """

    valid_score_modes = {"absolute", "positive", "negative"}
    valid_selection_sides = {"top", "bottom"}

    if config.score_mode not in valid_score_modes:
        raise ValueError(
            f"Unsupported score_mode='{config.score_mode}'. Expected one of {sorted(valid_score_modes)}"
        )
    if config.selection_side not in valid_selection_sides:
        raise ValueError(
            f"Unsupported selection_side='{config.selection_side}'. Expected one of {sorted(valid_selection_sides)}"
        )

    normalized_percent = _normalize_selection_percent(config.selection_percent)
    percent_token = _format_selection_percent_token(normalized_percent)
    return (
        f"importance_{task_name}_{config.score_mode}_"
        f"{config.selection_side}{percent_token}.pt"
    )


def resolve_precomputed_importance_paths(
    task_names: Sequence[str],
    config: PrecomputedImportanceConfig,
) -> Dict[str, Path]:
    """Resolve task-to-path mapping for precomputed importance artifacts.

    Args:
        task_names: Ordered task names to resolve.
        config: Filename configuration.

    Returns:
        Mapping of `task_name -> absolute path`.
    """

    resolved: Dict[str, Path] = {}
    for task_name in task_names:
        filename = build_precomputed_importance_filename(task_name, config)
        resolved[task_name] = config.root_dir / filename
    return resolved


IMPORTANCE_CFG = PrecomputedImportanceConfig(
    root_dir=Path(
        "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/"
        "Qwen3-1.7B-jwcm-v2-importance-only/importance"
    ),
    score_mode="absolute",
    selection_side="top",
    selection_percent=15.0,
)

TASK_NAMES: tuple[str, ...] = ("if", "math")
THRESHOLDS: tuple[float, ...] = (1e-8, 1e-7, 1e-6, 1e-5)
SAMPLED_VALUES_PER_TASK: int = 1_000_000
RANDOM_SEED: int = 42

importance_paths = resolve_precomputed_importance_paths(
    task_names=TASK_NAMES,
    config=IMPORTANCE_CFG,
)

# Validate all required files first so the analysis fails early with a clear message.
required_paths = [IMPORTANCE_CFG.root_dir, *importance_paths.values()]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Required path does not exist: {path}")

file_rows = []
for task_name, path in importance_paths.items():
    file_rows.append(
        {
            "task": task_name,
            "importance_path": str(path),
            "file_size_gib": float(path.stat().st_size / (1024 ** 3)),
        }
    )

importance_file_df = pd.DataFrame(file_rows).sort_values("task").reset_index(drop=True)
print("Resolved IF/Math importance files:")
display(importance_file_df)

## Experiment 2. 분석 유틸리티 함수 정의

In [ ]:
def set_global_seed(seed: int) -> None:
    """Set deterministic seeds for reproducibility.

    Args:
        seed: Integer random seed.

    Returns:
        None.
    """

    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_precomputed_importance_dictionary(
    task_name: str,
    importance_path: Path,
) -> Dict[str, torch.Tensor]:
    """Load one task's importance mapping and normalize tensors to CPU FP32.

    Args:
        task_name: Task key used for diagnostics.
        importance_path: Serialized `.pt` path.

    Returns:
        Mapping `parameter_name -> importance_tensor` in CPU FP32.

    Raises:
        FileNotFoundError: If the file does not exist.
        TypeError: If payload structure is invalid.
    """

    if not importance_path.exists():
        raise FileNotFoundError(
            f"Importance file for task '{task_name}' was not found: {importance_path}"
        )

    loaded_object = torch.load(importance_path, map_location="cpu")
    if not isinstance(loaded_object, Mapping):
        raise TypeError(
            f"Importance payload for task '{task_name}' must be a mapping, got {type(loaded_object)}"
        )

    normalized: Dict[str, torch.Tensor] = {}
    for parameter_name, tensor_value in loaded_object.items():
        if not torch.is_tensor(tensor_value):
            raise TypeError(
                f"Importance value must be tensor | task={task_name} | "
                f"parameter={parameter_name} | got={type(tensor_value)}"
            )

        # Force CPU FP32 so downstream statistics are explicit and consistent.
        normalized[str(parameter_name)] = tensor_value.detach().to(torch.float32).cpu()

    return normalized


def _apply_transform(absolute_values: torch.Tensor, transform_mode: str) -> torch.Tensor:
    """Apply value transform used in threshold/mass analysis.

    Args:
        absolute_values: Non-negative tensor values.
        transform_mode: `raw` or `sqrt`.

    Returns:
        Transformed tensor with same shape.

    Raises:
        ValueError: If transform mode is unsupported.
    """

    if transform_mode == "raw":
        return absolute_values
    if transform_mode == "sqrt":
        # Clamp is defensive: it avoids numerical issues if tiny negatives appear.
        return torch.sqrt(torch.clamp(absolute_values, min=0.0))
    raise ValueError(f"Unsupported transform_mode: {transform_mode}")


def _allocate_proportional_sample_counts(
    numel_by_parameter: Mapping[str, int],
    total_samples: int,
) -> Dict[str, int]:
    """Distribute a global sampling budget proportionally by tensor size.

    Args:
        numel_by_parameter: Mapping `parameter_name -> number_of_elements`.
        total_samples: Global sample budget.

    Returns:
        Mapping `parameter_name -> allocated sample count`.
    """

    if total_samples <= 0:
        return {}

    names = list(numel_by_parameter.keys())
    counts = np.asarray([max(int(numel_by_parameter[name]), 0) for name in names], dtype=np.float64)
    total_numel = float(counts.sum())
    if total_numel <= 0.0:
        return {}

    # Allocate base counts by floor, then distribute remaining budget by fractional part.
    raw_allocations = counts / total_numel * float(total_samples)
    base_allocations = np.floor(raw_allocations).astype(np.int64)
    remainder = int(total_samples - int(base_allocations.sum()))

    if remainder > 0:
        fractional = raw_allocations - base_allocations
        order = np.argsort(-fractional)
        for index in order[:remainder]:
            base_allocations[index] += 1

    allocated: Dict[str, int] = {}
    for name, allocation in zip(names, base_allocations):
        if allocation > 0:
            allocated[name] = int(allocation)
    return allocated


def collect_value_samples(
    importance_map: Mapping[str, torch.Tensor],
    total_samples: int,
    seed: int,
    transform_mode: str,
) -> np.ndarray:
    """Collect approximate global samples for visualization.

    Sampling is done proportionally per-parameter and uses random indices with replacement
    to avoid expensive full-length permutations on very large tensors.

    Args:
        importance_map: Mapping `parameter_name -> tensor`.
        total_samples: Approximate number of scalar values to sample.
        seed: RNG seed used for reproducible sampling.
        transform_mode: `raw` or `sqrt`.

    Returns:
        1D NumPy array of sampled transformed values.
    """

    numel_by_parameter = {
        parameter_name: int(tensor.numel())
        for parameter_name, tensor in importance_map.items()
        if int(tensor.numel()) > 0
    }
    sample_counts = _allocate_proportional_sample_counts(
        numel_by_parameter=numel_by_parameter,
        total_samples=total_samples,
    )

    rng = np.random.default_rng(seed)
    sampled_chunks: list[np.ndarray] = []

    for parameter_name, tensor in importance_map.items():
        sample_count = int(sample_counts.get(parameter_name, 0))
        if sample_count <= 0:
            continue

        flat_values = tensor.detach().reshape(-1)
        if flat_values.dtype != torch.float32:
            flat_values = flat_values.to(torch.float32)

        flat_numel = int(flat_values.numel())
        if sample_count >= flat_numel:
            sampled_values = flat_values.abs()
        else:
            sampled_indices = rng.integers(
                low=0,
                high=flat_numel,
                size=sample_count,
                endpoint=False,
                dtype=np.int64,
            )
            sampled_values = flat_values[torch.from_numpy(sampled_indices).to(torch.long)].abs()

        sampled_values = _apply_transform(sampled_values, transform_mode=transform_mode)
        sampled_chunks.append(sampled_values.cpu().numpy())

    if not sampled_chunks:
        return np.empty((0,), dtype=np.float32)

    return np.concatenate(sampled_chunks, axis=0).astype(np.float32, copy=False)


def summarize_threshold_mass(
    importance_map: Mapping[str, torch.Tensor],
    thresholds: Sequence[float],
    transform_mode: str,
    chunk_size: int = 8_000_000,
) -> pd.DataFrame:
    """Summarize active ratio and mass ratio for each threshold.

    The summary uses `value > threshold` for active-element definition and computes:
    - active_count / total_numel
    - active_mass / total_mass

    Chunked processing is intentionally used so the notebook can analyze large tensors
    without creating oversized temporary tensors for bucket indices.

    Args:
        importance_map: Mapping `parameter_name -> tensor`.
        thresholds: Positive thresholds to evaluate.
        transform_mode: `raw` or `sqrt`.
        chunk_size: Number of elements processed per chunk.

    Returns:
        DataFrame with one row per threshold.
    """

    sorted_thresholds = np.asarray(sorted(float(t) for t in thresholds), dtype=np.float64)
    threshold_tensor = torch.as_tensor(sorted_thresholds, dtype=torch.float32)

    # Bin 0: <= t0, Bin i: (t_{i-1}, t_i], Bin n: > t_{n-1}
    number_of_bins = int(len(sorted_thresholds) + 1)
    count_bins = torch.zeros(number_of_bins, dtype=torch.float64)
    mass_bins = torch.zeros(number_of_bins, dtype=torch.float64)

    total_numel = 0
    total_mass = 0.0

    safe_chunk_size = max(int(chunk_size), 1)

    for tensor in importance_map.values():
        flat_values = tensor.detach().reshape(-1)
        if flat_values.dtype != torch.float32:
            flat_values = flat_values.to(torch.float32)

        flat_numel = int(flat_values.numel())
        for start_index in range(0, flat_numel, safe_chunk_size):
            end_index = min(start_index + safe_chunk_size, flat_numel)
            absolute_chunk = flat_values[start_index:end_index].abs()
            transformed_chunk = _apply_transform(absolute_chunk, transform_mode=transform_mode)

            bucket_indices = torch.bucketize(transformed_chunk, boundaries=threshold_tensor, right=True)
            count_bins += torch.bincount(bucket_indices, minlength=number_of_bins).to(torch.float64)
            mass_bins += torch.bincount(
                bucket_indices,
                weights=transformed_chunk.to(torch.float64),
                minlength=number_of_bins,
            )

            total_numel += int(transformed_chunk.numel())
            total_mass += float(transformed_chunk.sum().item())

    # Descending cumulative sums allow O(1) retrieval of `> threshold` stats per threshold.
    count_cumsum_desc = torch.flip(torch.cumsum(torch.flip(count_bins, dims=[0]), dim=0), dims=[0])
    mass_cumsum_desc = torch.flip(torch.cumsum(torch.flip(mass_bins, dims=[0]), dim=0), dims=[0])

    rows: list[dict[str, Any]] = []
    for threshold_index, threshold_value in enumerate(sorted_thresholds):
        # For threshold k, active bins are [k+1, ..., n].
        active_count = int(round(float(count_cumsum_desc[threshold_index + 1].item())))
        active_mass = float(mass_cumsum_desc[threshold_index + 1].item())

        rows.append(
            {
                "threshold": float(threshold_value),
                "total_numel": int(total_numel),
                "active_count": int(active_count),
                "active_ratio": float(active_count / max(total_numel, 1)),
                "total_mass": float(total_mass),
                "active_mass": float(active_mass),
                "active_mass_ratio": float(active_mass / total_mass) if total_mass > 0.0 else 0.0,
            }
        )

    return pd.DataFrame(rows)


def run_task_analysis(
    task_name: str,
    importance_path: Path,
    thresholds: Sequence[float],
    total_samples: int,
    seed: int,
) -> Dict[str, Any]:
    """Run raw/sqrt threshold analysis and sample extraction for one task.

    Args:
        task_name: Task identifier.
        importance_path: Serialized importance path.
        thresholds: Thresholds for active-ratio analysis.
        total_samples: Number of sampled values for distribution plots.
        seed: Reproducible RNG seed.

    Returns:
        Dictionary containing summary DataFrames and sampled arrays.
    """

    importance_map = load_precomputed_importance_dictionary(
        task_name=task_name,
        importance_path=importance_path,
    )

    number_of_tensors = int(len(importance_map))

    raw_summary_df = summarize_threshold_mass(
        importance_map=importance_map,
        thresholds=thresholds,
        transform_mode="raw",
    )
    raw_summary_df.insert(0, "task", task_name)
    raw_summary_df.insert(1, "transform", "raw")

    sqrt_summary_df = summarize_threshold_mass(
        importance_map=importance_map,
        thresholds=thresholds,
        transform_mode="sqrt",
    )
    sqrt_summary_df.insert(0, "task", task_name)
    sqrt_summary_df.insert(1, "transform", "sqrt")

    raw_samples = collect_value_samples(
        importance_map=importance_map,
        total_samples=total_samples,
        seed=seed,
        transform_mode="raw",
    )
    # Reuse the same sampled points for sqrt to keep raw-vs-sqrt comparisons aligned.
    sqrt_samples = np.sqrt(np.clip(raw_samples, a_min=0.0, a_max=None))

    total_numel = int(raw_summary_df["total_numel"].iloc[0])

    # Explicit cleanup is important here because each importance file is multi-GB.
    del importance_map
    gc.collect()

    return {
        "task": task_name,
        "num_tensors": number_of_tensors,
        "total_numel": total_numel,
        "raw_summary_df": raw_summary_df,
        "sqrt_summary_df": sqrt_summary_df,
        "raw_samples": raw_samples,
        "sqrt_samples": sqrt_samples,
    }

## Experiment 3. IF/Math score 로드 + threshold/mass 요약 계산

In [ ]:
set_global_seed(RANDOM_SEED)

analysis_by_task: Dict[str, Dict[str, Any]] = {}
for index, (task_name, importance_path) in enumerate(importance_paths.items()):
    # Offset seed per task for deterministic but task-distinct sampling.
    analysis_by_task[task_name] = run_task_analysis(
        task_name=task_name,
        importance_path=importance_path,
        thresholds=THRESHOLDS,
        total_samples=SAMPLED_VALUES_PER_TASK,
        seed=RANDOM_SEED + index,
    )

raw_summary_df = pd.concat(
    [analysis_by_task[task_name]["raw_summary_df"] for task_name in TASK_NAMES],
    axis=0,
    ignore_index=True,
).sort_values(["task", "threshold"]).reset_index(drop=True)

sqrt_summary_df = pd.concat(
    [analysis_by_task[task_name]["sqrt_summary_df"] for task_name in TASK_NAMES],
    axis=0,
    ignore_index=True,
).sort_values(["task", "threshold"]).reset_index(drop=True)

basic_stats_rows = []
for task_name in TASK_NAMES:
    task_payload = analysis_by_task[task_name]
    basic_stats_rows.append(
        {
            "task": task_name,
            "num_tensors": int(task_payload["num_tensors"]),
            "total_numel": int(task_payload["total_numel"]),
            "sampled_values": int(len(task_payload["raw_samples"])),
        }
    )

basic_stats_df = pd.DataFrame(basic_stats_rows).sort_values("task").reset_index(drop=True)

print("Per-task loaded tensor statistics:")
display(basic_stats_df)

print("Raw score threshold summary:")
display(raw_summary_df)

print("Sqrt(score) threshold summary:")
display(sqrt_summary_df)

raw_ratio_pivot = raw_summary_df.pivot(index="threshold", columns="task", values="active_ratio")
raw_mass_pivot = raw_summary_df.pivot(index="threshold", columns="task", values="active_mass_ratio")
sqrt_ratio_pivot = sqrt_summary_df.pivot(index="threshold", columns="task", values="active_ratio")
sqrt_mass_pivot = sqrt_summary_df.pivot(index="threshold", columns="task", values="active_mass_ratio")

print("Raw active ratio pivot:")
display(raw_ratio_pivot)
print("Raw active mass ratio pivot:")
display(raw_mass_pivot)
print("Sqrt active ratio pivot:")
display(sqrt_ratio_pivot)
print("Sqrt active mass ratio pivot:")
display(sqrt_mass_pivot)

## Experiment 4. 시각화 함수 정의 (축 범위 최적화 포함)

In [ ]:
def compute_zoomed_ylim(
    values: Sequence[float],
    lower_bound: float,
    upper_bound: float,
    padding_ratio: float = 0.15,
    min_span: float = 0.02,
) -> tuple[float, float]:
    """Compute a zoomed y-axis range that still preserves visual stability.

    Args:
        values: Numeric values plotted on y-axis.
        lower_bound: Hard lower clipping bound.
        upper_bound: Hard upper clipping bound.
        padding_ratio: Relative padding applied around observed min/max.
        min_span: Minimum y-span to avoid over-zooming.

    Returns:
        `(y_min, y_max)` tuple for plotting.
    """

    array = np.asarray(values, dtype=np.float64)
    if array.size == 0:
        return float(lower_bound), float(upper_bound)

    min_value = float(np.nanmin(array))
    max_value = float(np.nanmax(array))
    span = max(max_value - min_value, min_span)
    padding = span * padding_ratio

    y_min = max(lower_bound, min_value - padding)
    y_max = min(upper_bound, max_value + padding)

    if y_max <= y_min:
        midpoint = float((min_value + max_value) * 0.5)
        half_span = min_span * 0.5
        y_min = max(lower_bound, midpoint - half_span)
        y_max = min(upper_bound, midpoint + half_span)

    return float(y_min), float(y_max)


def plot_threshold_metric_panels(summary_df: pd.DataFrame, title_prefix: str) -> None:
    """Plot threshold-vs-ratio trends with log-scaled x and zoomed y-ranges.

    Args:
        summary_df: Summary DataFrame with ratio columns.
        title_prefix: Prefix used in subplot titles.

    Returns:
        None.
    """

    ordered = summary_df.sort_values(["task", "threshold"]).reset_index(drop=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

    metric_specs = [
        ("active_ratio", "Active Parameter Ratio", axes[0]),
        ("active_mass_ratio", "Active Mass Ratio", axes[1]),
    ]

    for metric_name, y_label, axis in metric_specs:
        for task_name in sorted(ordered["task"].unique()):
            task_slice = ordered[ordered["task"] == task_name]
            axis.plot(
                task_slice["threshold"],
                task_slice[metric_name],
                marker="o",
                linewidth=2.0,
                label=task_name,
            )

        axis.set_xscale("log")
        axis.set_xlim(min(THRESHOLDS), max(THRESHOLDS))
        axis.set_xticks(THRESHOLDS)
        axis.set_xlabel("Threshold")
        axis.set_ylabel(y_label)
        axis.set_title(f"{title_prefix} | {y_label} (Zoomed)")
        axis.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))

        y_min, y_max = compute_zoomed_ylim(
            values=ordered[metric_name].to_numpy(),
            lower_bound=0.0,
            upper_bound=1.0,
            padding_ratio=0.15,
            min_span=0.02,
        )
        axis.set_ylim(y_min, y_max)
        axis.grid(True, which="both", linestyle="--", alpha=0.35)
        axis.legend(title="Task")

    plt.show()


def plot_active_count_bars(summary_df: pd.DataFrame, title_prefix: str) -> None:
    """Plot threshold-wise active parameter counts in billions.

    Args:
        summary_df: Summary DataFrame with `active_count`.
        title_prefix: Prefix used in the chart title.

    Returns:
        None.
    """

    ordered = summary_df.sort_values(["threshold", "task"]).copy()
    ordered["active_count_billion"] = ordered["active_count"] / 1e9
    ordered["threshold_label"] = ordered["threshold"].map(lambda value: f"{value:.0e}")

    if HAVE_SEABORN:
        plt.figure(figsize=(12, 5))
        axis = sns.barplot(
            data=ordered,
            x="threshold_label",
            y="active_count_billion",
            hue="task",
            palette="deep",
        )
    else:
        # Matplotlib fallback to keep grouped bars available without seaborn.
        threshold_labels = [f"{value:.0e}" for value in sorted(ordered["threshold"].unique())]
        task_names = sorted(ordered["task"].unique())
        pivot = ordered.pivot(index="threshold_label", columns="task", values="active_count_billion")
        pivot = pivot.reindex(index=threshold_labels, columns=task_names)

        x_positions = np.arange(len(threshold_labels), dtype=np.float64)
        number_of_tasks = max(len(task_names), 1)
        bar_width = 0.8 / number_of_tasks

        fig, axis = plt.subplots(figsize=(12, 5))
        for task_index, task_name in enumerate(task_names):
            offsets = x_positions - 0.4 + bar_width * (task_index + 0.5)
            axis.bar(offsets, pivot[task_name].to_numpy(), width=bar_width, label=task_name)

        axis.set_xticks(x_positions)
        axis.set_xticklabels(threshold_labels)

    axis.set_title(f"{title_prefix} | Active Parameter Count by Threshold")
    axis.set_xlabel("Threshold")
    axis.set_ylabel("Active Parameters (Billions)")

    max_value = float(ordered["active_count_billion"].max()) if len(ordered) > 0 else 0.0
    axis.set_ylim(0.0, max(max_value * 1.12, 0.01))
    axis.grid(True, axis="y", linestyle="--", alpha=0.35)
    axis.legend(title="Task")
    plt.tight_layout()
    plt.show()


def plot_log_histograms(
    sampled_values_by_task: Mapping[str, np.ndarray],
    thresholds: Sequence[float],
    title_prefix: str,
    transform_mode: str,
) -> None:
    """Plot log10-distribution curves for sampled values with threshold markers.

    Args:
        sampled_values_by_task: Mapping `task -> sampled values`.
        thresholds: Thresholds to visualize as vertical lines.
        title_prefix: Prefix used in the chart title.
        transform_mode: `raw` or `sqrt`.

    Returns:
        None.
    """

    transformed_logs: Dict[str, np.ndarray] = {}
    for task_name, sampled_values in sampled_values_by_task.items():
        safe_values = np.maximum(np.asarray(sampled_values, dtype=np.float64), 1e-30)
        transformed_logs[task_name] = np.log10(safe_values)

    all_logs = np.concatenate(list(transformed_logs.values()), axis=0)
    lower = float(np.percentile(all_logs, 0.5))
    upper = float(np.percentile(all_logs, 99.5))
    if not np.isfinite(lower) or not np.isfinite(upper) or upper <= lower:
        lower, upper = float(np.min(all_logs)), float(np.max(all_logs))

    number_of_bins = 140
    max_density = 0.0
    histogram_cache: Dict[str, tuple[np.ndarray, np.ndarray]] = {}

    for task_name, log_values in transformed_logs.items():
        density, bin_edges = np.histogram(
            log_values,
            bins=number_of_bins,
            range=(lower, upper),
            density=True,
        )
        histogram_cache[task_name] = (density, bin_edges)
        max_density = max(max_density, float(np.nanmax(density)) if density.size > 0 else 0.0)

    plt.figure(figsize=(12, 5))
    for task_name, (density, bin_edges) in histogram_cache.items():
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) * 0.5
        plt.plot(bin_centers, density, linewidth=2.0, label=task_name)

    for threshold in thresholds:
        plt.axvline(
            x=np.log10(float(threshold)),
            color="gray",
            linestyle="--",
            linewidth=1.0,
            alpha=0.35,
        )

    x_label = "log10(|S|)" if transform_mode == "raw" else "log10(sqrt(|S|))"
    plt.title(f"{title_prefix} | Sampled Value Distribution ({x_label})")
    plt.xlabel(x_label)
    plt.ylabel("Density")
    plt.xlim(lower, upper)
    plt.ylim(0.0, max(max_density * 1.10, 0.01))
    plt.grid(True, linestyle="--", alpha=0.35)
    plt.legend(title="Task")
    plt.tight_layout()
    plt.show()


def display_metric_pivot(summary_df: pd.DataFrame, metric: str, title: str) -> None:
    """Display a threshold-by-task pivot table for a selected metric.

    Args:
        summary_df: Summary DataFrame.
        metric: Metric column to pivot.
        title: Header message printed before table display.

    Returns:
        None.
    """

    pivot = summary_df.pivot(index="threshold", columns="task", values=metric)
    print(title)
    display(pivot)

## Experiment 5. Raw score 분석 시각화

In [ ]:
raw_samples_by_task = {
    task_name: analysis_by_task[task_name]["raw_samples"] for task_name in TASK_NAMES
}

plot_threshold_metric_panels(raw_summary_df, title_prefix="Raw |S|")
plot_active_count_bars(raw_summary_df, title_prefix="Raw |S|")
plot_log_histograms(
    sampled_values_by_task=raw_samples_by_task,
    thresholds=THRESHOLDS,
    title_prefix="Raw |S|",
    transform_mode="raw",
)

display_metric_pivot(
    summary_df=raw_summary_df,
    metric="active_ratio",
    title="Raw | Active parameter ratio pivot",
)
display_metric_pivot(
    summary_df=raw_summary_df,
    metric="active_mass_ratio",
    title="Raw | Active mass ratio pivot",
)

## Experiment 6. Sqrt(score) 분석 시각화

In [ ]:
sqrt_samples_by_task = {
    task_name: analysis_by_task[task_name]["sqrt_samples"] for task_name in TASK_NAMES
}

plot_threshold_metric_panels(sqrt_summary_df, title_prefix="Sqrt(|S|)")
plot_active_count_bars(sqrt_summary_df, title_prefix="Sqrt(|S|)")
plot_log_histograms(
    sampled_values_by_task=sqrt_samples_by_task,
    thresholds=THRESHOLDS,
    title_prefix="Sqrt(|S|)",
    transform_mode="sqrt",
)

display_metric_pivot(
    summary_df=sqrt_summary_df,
    metric="active_ratio",
    title="Sqrt(|S|) | Active parameter ratio pivot",
)
display_metric_pivot(
    summary_df=sqrt_summary_df,
    metric="active_mass_ratio",
    title="Sqrt(|S|) | Active mass ratio pivot",
)

## Experiment 7. Raw vs Sqrt 비교 시각화

In [ ]:
comparison_df = pd.concat(
    [raw_summary_df.assign(transform_label="raw"), sqrt_summary_df.assign(transform_label="sqrt")],
    axis=0,
    ignore_index=True,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
metric_specs = [
    ("active_ratio", "Active Parameter Ratio"),
    ("active_mass_ratio", "Active Mass Ratio"),
]

for axis, (metric_name, metric_title) in zip(axes, metric_specs):
    for task_name in TASK_NAMES:
        for transform_label, line_style in (("raw", "-"), ("sqrt", "--")):
            subset = comparison_df[
                (comparison_df["task"] == task_name)
                & (comparison_df["transform_label"] == transform_label)
            ].sort_values("threshold")

            axis.plot(
                subset["threshold"],
                subset[metric_name],
                marker="o",
                linestyle=line_style,
                linewidth=2.0,
                label=f"{task_name}-{transform_label}",
            )

    axis.set_xscale("log")
    axis.set_xlim(min(THRESHOLDS), max(THRESHOLDS))
    axis.set_xticks(THRESHOLDS)
    axis.set_xlabel("Threshold")
    axis.set_ylabel(metric_title)
    axis.set_title(f"Raw vs Sqrt | {metric_title}")
    axis.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))

    y_min, y_max = compute_zoomed_ylim(
        values=comparison_df[metric_name].to_numpy(),
        lower_bound=0.0,
        upper_bound=1.0,
        padding_ratio=0.15,
        min_span=0.02,
    )
    axis.set_ylim(y_min, y_max)
    axis.grid(True, which="both", linestyle="--", alpha=0.35)
    axis.legend(fontsize=9)

plt.show()

merged_comparison = raw_summary_df[
    ["task", "threshold", "active_ratio", "active_mass_ratio"]
].merge(
    sqrt_summary_df[["task", "threshold", "active_ratio", "active_mass_ratio"]],
    on=["task", "threshold"],
    suffixes=("_raw", "_sqrt"),
)
merged_comparison["active_ratio_delta_sqrt_minus_raw"] = (
    merged_comparison["active_ratio_sqrt"] - merged_comparison["active_ratio_raw"]
)
merged_comparison["active_mass_ratio_delta_sqrt_minus_raw"] = (
    merged_comparison["active_mass_ratio_sqrt"] - merged_comparison["active_mass_ratio_raw"]
)

print("Raw vs Sqrt delta table:")
display(merged_comparison.sort_values(["task", "threshold"]).reset_index(drop=True))